# Aggregate the data from the ROBIN simulator

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import pandas as pd


## Set parameters and load the data

In [3]:
path_kernel_output = '../data/MAD-BCN/kernel_output'
kernel_output_file = 'output_with_service.csv'
path_prices_data = '../data/MAD-BCN/prices/prices_ext_MAD-BCN_2025.csv'
path_aggregated_data = '../data/MAD-BCN/aggregated/aggregated_MAD-BCN_2025.csv'
path_final_data = '../data/MAD-BCN/aggregated/MAD-BCN_2025.csv'
ref_provider = ['AVE', 'AVLO']
path_final_data_ref_provider = '../data/MAD-BCN/aggregated/MAD-BCN_2025_RENFE.csv'

figures_path = '../reports/figures/MAD-BCN'

os.makedirs(figures_path, exist_ok=True)


In [4]:
prices_data = pd.read_csv(path_prices_data)
prices_data.head()

,trip_id,origin,destination,tsp,train_type,departure,arrival,duration,service_id,capacity,train_model,BasicSeat
0,1,60000,71801,AVLO,AVLO,2025-01-01 06:12:00,2025-01-01 08:49:00,0 days 02:37:00,00001_01-01-2025-06.12,581.0,S-106,34.52
1,2,60000,71801,IRYO,IRYO,2025-01-01 06:22:00,2025-01-01 08:59:00,0 days 02:37:00,00002_01-01-2025-06.22,461.0,S-109,43.16
2,3,60000,71801,AVE,AVE,2025-01-01 06:27:00,2025-01-01 09:25:00,0 days 02:58:00,00003_01-01-2025-06.27,359.0,S-103,55.70
3,4,60000,71801,AVE,AVE,2025-01-01 06:57:00,2025-01-01 09:34:00,0 days 02:37:00,00004_01-01-2025-06.57,414.0,S-103,82.99
4,5,60000,71801,OUIGO,OUIGO,2025-01-01 07:02:00,2025-01-01 09:54:00,0 days 02:52:00,00005_01-01-2025-07.02,509.0,S-108,36.73


In [5]:
kernel_output = pd.read_csv(os.path.join(path_kernel_output, kernel_output_file))

kernel_output

,id,user_pattern,departure_station,arrival_station,arrival_day,arrival_time,purchase_date,service,service_departure_time,service_arrival_time,seat,price,utility,best_service,best_seat,best_utility,service_id,train_type
0,10788,Leisure,60000,71801,2025-01-01,13.841214,2024-12-06,00009_01-01-2025-12.22,12.366667,15.233333,BasicSeat,29.04,8.801841,00009_01-01-2025-12.22,BasicSeat,8.801841,00009_01-01-2025-12.22,IRYO
1,10660,Leisure,60000,71801,2025-01-01,8.487919,2024-12-08,00009_01-01-2025-17.22,17.366667,20.233333,BasicSeat,25.63,8.741205,00009_01-01-2025-17.22,BasicSeat,8.741205,00009_01-01-2025-17.22,IRYO
2,9333,Leisure,60000,71801,2025-01-01,12.059012,2024-12-09,00019_01-01-2025-19.34,19.566667,22.916667,BasicSeat,32.14,8.868130,00019_01-01-2025-19.34,BasicSeat,8.868130,00019_01-01-2025-19.34,AVLO
3,10725,Leisure,60000,71801,2025-01-01,14.041513,2024-12-09,00018_01-01-2025-15.27,15.450000,18.766667,BasicSeat,29.46,8.613906,00018_01-01-2025-15.27,BasicSeat,8.613906,00018_01-01-2025-15.27,AVLO
4,11646,Leisure,60000,71801,2025-01-01,14.190313,2024-12-09,00006_01-01-2025-07.27,7.450000,10.766667,BasicSeat,27.76,8.151557,00006_01-01-2025-07.27,BasicSeat,8.151557,00006_01-01-2025-07.27,AVE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4413527,4413512,Business-Morning,60000,71801,2025-12-31,7.448939,2025-12-31,00002_31-12-2025-07.22,7.366667,9.983333,BasicSeat,24.33,1.990050,00002_31-12-2025-07.22,BasicSeat,1.990050,00002_31-12-2025-07.22,IRYO
4413528,4413515,Business-Evening,60000,71801,2025-12-31,18.690937,2025-12-31,00004_31-12-2025-15.57,15.950000,18.566667,BasicSeat,46.19,11.004887,00004_31-12-2025-15.57,BasicSeat,11.004887,00004_31-12-2025-15.57,AVE
4413529,4413526,Business-Morning,60000,71801,2025-12-31,7.309182,2025-12-31,00005_31-12-2025-07.02,7.033333,9.900000,BasicSeat,48.91,1.050472,00005_31-12-2025-07.02,BasicSeat,1.050472,00005_31-12-2025-07.02,OUIGO
4413530,4413528,Business-Morning,60000,71801,2025-12-31,8.027549,2025-12-31,00002_31-12-2025-06.22,6.366667,8.983333,BasicSeat,24.29,1.201989,00002_31-12-2025-06.22,BasicSeat,1.201989,00002_31-12-2025-06.22,IRYO


## Aggregated dataset

In [6]:
# Dataset columns
# - service_id
# - train_type
# - year
# - month
# - day_of_week
# - departure_time
# - duration
# - price
# - passengers

# Other columns that could be added:
# - Min/max/avg ticket purchase date
# - arrival_time
# - Weekday or weekend

# - Regional holiday or similar to indicate high-demand periods

In [7]:
# Remove passengers that do not travel, i.e. 'service_id' is NaN
n_rows = kernel_output.shape[0]
kernel_output = kernel_output[kernel_output['service_id'].notna()]

print(f"Number of travelers out of the whole dataset: {kernel_output.shape[0]} / {n_rows}")

Number of travelers out of the whole dataset: 4281106 / 4413532


In [8]:
# Consider that the passengers on 'AVE INT' are equivalent to 'AVE'
kernel_output['train_type'] = kernel_output['train_type'].replace('AVE INT', 'AVE')
prices_data['train_type'] = prices_data['train_type'].replace('AVE INT', 'AVE')
kernel_output['train_type'].unique()

/var/folders/n5/s8_w_cgn3jd6kv8h7psy1jnr0000gn/T/ipykernel_26011/3347484802.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  kernel_output['train_type'] = kernel_output['train_type'].replace('AVE INT', 'AVE')


array(['IRYO', 'AVLO', 'AVE', 'OUIGO'], dtype=object)

In [9]:
# Create the new dataframe
agg_dataset = pd.DataFrame()

agg_dataset['service_id'] = prices_data['service_id']
agg_dataset['train_type'] = prices_data['train_type']
agg_dataset['capacity'] = prices_data['capacity'].astype(int)
agg_dataset['train_model'] = prices_data['train_model']

departure_dt = pd.to_datetime(prices_data['departure'], format='%Y-%m-%d %H:%M:%S')
arrival_dt = pd.to_datetime(prices_data['arrival'], format='%Y-%m-%d %H:%M:%S')
agg_dataset['year'] = departure_dt.dt.year
agg_dataset['month'] = departure_dt.dt.month
agg_dataset['day_of_week'] = departure_dt.dt.dayofweek
agg_dataset['departure_time'] = round(departure_dt.dt.hour + departure_dt.dt.minute / 60 + departure_dt.dt.second / 3600, 2)

duration_dt = arrival_dt - departure_dt
agg_dataset['duration'] = duration_dt.dt.total_seconds() / 60

agg_dataset['price'] = prices_data['BasicSeat']

# Count the number of passengers per service_id
kernel_output_grouped = kernel_output.groupby('service_id').size()
agg_dataset['passengers'] = agg_dataset['service_id'].map(kernel_output_grouped).fillna(0)
agg_dataset['passengers'] = agg_dataset['passengers'].astype(int)

print(f"Number of travelers in the agg_dataset: {agg_dataset['passengers'].sum()} / {n_rows}")
print(f"Number of services with passengers in the agg_dataset: {kernel_output['service_id'].nunique()} / {agg_dataset['service_id'].nunique()}")

# Save the agg_dataset
agg_dataset.to_csv(path_aggregated_data, index=False)
print(f"agg_dataset saved to {path_aggregated_data}")


Number of travelers in the agg_dataset: 4281106 / 4413532
Number of services with passengers in the agg_dataset: 14296 / 15438
agg_dataset saved to ../data/MAD-BCN/aggregated/aggregated_MAD-BCN_2025.csv


In [10]:
agg_dataset

,service_id,train_type,capacity,train_model,year,month,day_of_week,departure_time,duration,price,passengers
0,00001_01-01-2025-06.12,AVLO,581,S-106,2025,1,2,6.20,157.0,34.52,559
1,00002_01-01-2025-06.22,IRYO,461,S-109,2025,1,2,6.37,157.0,43.16,220
2,00003_01-01-2025-06.27,AVE,359,S-103,2025,1,2,6.45,178.0,55.70,180
3,00004_01-01-2025-06.57,AVE,414,S-103,2025,1,2,6.95,157.0,82.99,13
4,00005_01-01-2025-07.02,OUIGO,509,S-108,2025,1,2,7.03,172.0,36.73,50
...,...,...,...,...,...,...,...,...,...,...,...
15433,00004_31-12-2025-20.05,AVE,414,S-103,2025,12,2,20.08,157.0,21.42,1823
15434,00009_31-12-2025-20.22,IRYO,461,S-109,2025,12,2,20.37,172.0,31.22,10
15435,00020_31-12-2025-20.27,AVE,414,S-103,2025,12,2,20.45,201.0,55.58,0
15436,00005_31-12-2025-21.02,OUIGO,509,S-108,2025,12,2,21.03,172.0,24.94,122


In [11]:
display(f"Percentage of passengers in relation to the total capacity of the services: {round(agg_dataset['passengers'].sum()/agg_dataset['capacity'].sum()*100, 2)} %")

'Percentage of passengers in relation to the total capacity of the services: 59.46 %'

## Final dataset

In [12]:
# Same dataset, adding columns for each competing service (2 before and 2 after):
# - Time difference (negative if earlier)
# - Competitor's price
# - Company
# - Trip duration

In [13]:
final_dataset = agg_dataset.copy()
competing_range = [-2, 2]
# For each service include the information about competing services (2 before and 2 after):
for i in range(competing_range[0], competing_range[1]+1):
    if i == 0:
        continue
    # Get the service_id of the service that competes
    #final_dataset[f'service_id_competitor_{i}'] = final_dataset['service_id'].shift(-i)

    # Get the temporal distance (negative if it is before)
    time_distance = final_dataset['departure_time'].shift(-i) - final_dataset['departure_time']
    if i < 0:
        # Means that the service is before, so if the distance is positive, means the train was the day before
        time_distance[time_distance > 0] = -time_distance[time_distance > 0]
    elif i > 0:
        # Means that the service is after, so if the distance is negative, means the train was the day after
        time_distance[time_distance < 0] = -time_distance[time_distance < 0]
    final_dataset[f'time_distance_competitor_{i}'] = round(time_distance * 60, 2)

    # Get the price of the competitor
    final_dataset[f'price_competitor_{i}'] = final_dataset['price'].shift(-i)

    # Get the train type of the competitor
    final_dataset[f'train_type_competitor_{i}'] = final_dataset['train_type'].shift(-i)

    # Get the duration of the competitor
    final_dataset[f'duration_competitor_{i}'] = final_dataset['duration'].shift(-i)


# Count and remove the rows with NaN values
n_rows = final_dataset.shape[0]
final_dataset = final_dataset.dropna()
print(f"There have been {n_rows - final_dataset.shape[0]} rows with NaN values removed")

# Save the final dataset
final_dataset.to_csv(path_final_data, index=False)
print(f"final_dataset saved to {path_final_data}")


There have been 368 rows with NaN values removed
final_dataset saved to ../data/MAD-BCN/aggregated/MAD-BCN_2025.csv


In [14]:
final_dataset

,service_id,train_type,capacity,train_model,year,month,day_of_week,departure_time,duration,price,...,train_type_competitor_-1,duration_competitor_-1,time_distance_competitor_1,price_competitor_1,train_type_competitor_1,duration_competitor_1,time_distance_competitor_2,price_competitor_2,train_type_competitor_2,duration_competitor_2
2,00003_01-01-2025-06.27,AVE,359,S-103,2025,1,2,6.45,178.0,55.70,...,IRYO,157.0,30.0,82.99,AVE,157.0,34.8,36.73,OUIGO,172.0
3,00004_01-01-2025-06.57,AVE,414,S-103,2025,1,2,6.95,157.0,82.99,...,AVE,178.0,4.8,36.73,OUIGO,172.0,25.2,30.65,IRYO,157.0
4,00005_01-01-2025-07.02,OUIGO,509,S-108,2025,1,2,7.03,172.0,36.73,...,AVE,157.0,20.4,30.65,IRYO,157.0,25.2,27.76,AVE,199.0
5,00002_01-01-2025-07.22,IRYO,461,S-109,2025,1,2,7.37,157.0,30.65,...,OUIGO,172.0,4.8,27.76,AVE,199.0,34.8,47.82,AVE,157.0
6,00006_01-01-2025-07.27,AVE,414,S-103,2025,1,2,7.45,199.0,27.76,...,IRYO,157.0,30.0,47.82,AVE,157.0,60.0,104.56,AVE,172.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15431,00009_31-12-2025-19.29,IRYO,461,S-109,2025,12,2,19.48,172.0,34.28,...,AVE,157.0,5.4,25.68,AVLO,201.0,36.0,21.42,AVE,157.0
15432,00019_31-12-2025-19.34,AVLO,581,S-106,2025,12,2,19.57,201.0,25.68,...,IRYO,172.0,30.6,21.42,AVE,157.0,48.0,31.22,IRYO,172.0
15433,00004_31-12-2025-20.05,AVE,414,S-103,2025,12,2,20.08,157.0,21.42,...,AVLO,201.0,17.4,31.22,IRYO,172.0,22.2,55.58,AVE,201.0
15434,00009_31-12-2025-20.22,IRYO,461,S-109,2025,12,2,20.37,172.0,31.22,...,AVE,157.0,4.8,55.58,AVE,201.0,39.6,24.94,OUIGO,172.0


In [15]:
# Final dataset only with the prices of the reference provider

final_dataset_ref_provider = agg_dataset.copy()
competing_range = [-2, 2]

# Initialize the columns for the competitors
for i in range(competing_range[0], competing_range[1] + 1):
    if i == 0:
        continue
    # final_dataset_ref_provider[f'service_id_competitor_{i}'] = None
    final_dataset_ref_provider[f'time_distance_competitor_{i}'] = None
    final_dataset_ref_provider[f'price_competitor_{i}'] = None
    final_dataset_ref_provider[f'train_type_competitor_{i}'] = None
    final_dataset_ref_provider[f'duration_competitor_{i}'] = None

# Filter the dataset to only include the reference provider
for row_idx in range(final_dataset_ref_provider.shape[0]):
    row = final_dataset_ref_provider.iloc[row_idx]

    iteration_range = [-1, 1]
    i = 0
    # For each service add the information of the previous services of the reference provider

    # Explore the competing_range[0] previous services in ref_provider and competing_range[1] next services in ref_provider, if not in service_provider_do not count this one
    while iteration_range[0] >= competing_range[0] or iteration_range[1] <= competing_range[1]:
        if iteration_range[0] >= competing_range[0]:
            # Explore previous services
            i -= 1
        elif iteration_range[1] <= competing_range[1]:
            if i < 0:
                i = 0
            # Explore next services
            i += 1

        # Check if the index is out of bounds and skip if it is
        if row_idx + i < 0 or row_idx + i >= final_dataset_ref_provider.shape[0]:
            iteration_range[0] -= 1 if i < 0 else 0
            iteration_range[1] += 1 if i > 0 else 0
            continue

        # Check if the train type is in the reference provider
        if final_dataset_ref_provider.iloc[row_idx + i]['train_type'] not in ref_provider:
            continue
   
        rel_i = iteration_range[0] if i < 0 else iteration_range[1]

        # Get the service_id of the service that competes
        #final_dataset_ref_provider.at[row_idx, f'service_id_competitor_{rel_i}'] = final_dataset_ref_provider.iloc[row_idx + i]['service_id']

        # Get the temporal distance (negative if it is before)
        time_distance = final_dataset_ref_provider.iloc[row_idx + i]['departure_time'] - row['departure_time']
        if i < 0:
            # Means that the service is before, so if the distance is positive, means the train was the day before
            if time_distance > 0:
                time_distance = -time_distance
        elif i > 0:
            # Means that the service is after, so if the distance is negative, means the train was the day after
            if time_distance < 0:
                time_distance = -time_distance
        final_dataset_ref_provider.at[row_idx, f'time_distance_competitor_{rel_i}'] = round(time_distance * 60, 2)
        
        # Get the price of the competitor
        final_dataset_ref_provider.at[row_idx, f'price_competitor_{rel_i}'] = final_dataset_ref_provider.iloc[row_idx + i]['price']

        # Get the train type of the competitor
        final_dataset_ref_provider.at[row_idx, f'train_type_competitor_{rel_i}'] = final_dataset_ref_provider.iloc[row_idx + i]['train_type']
                
        # Get the duration of the competitor
        final_dataset_ref_provider.at[row_idx, f'duration_competitor_{rel_i}'] = final_dataset_ref_provider.iloc[row_idx + i]['duration']

        # Increment the iteration range
        iteration_range[0] -= 1 if i < 0 else 0
        iteration_range[1] += 1 if i > 0 else 0
        

# Count and remove the rows with NaN values
n_rows = final_dataset_ref_provider.shape[0]
final_dataset_ref_provider = final_dataset_ref_provider.dropna()
print(f"There have been {n_rows - final_dataset_ref_provider.shape[0]} rows with NaN values removed")

# Save the final dataset
final_dataset_ref_provider.to_csv(path_final_data_ref_provider, index=False)
print(f"final_dataset_ref_provider saved to {path_final_data_ref_provider}")



There have been 422 rows with NaN values removed
final_dataset_ref_provider saved to ../data/MAD-BCN/aggregated/MAD-BCN_2025_RENFE.csv


In [16]:
final_dataset_ref_provider

,service_id,train_type,capacity,train_model,year,month,day_of_week,departure_time,duration,price,...,train_type_competitor_-1,duration_competitor_-1,time_distance_competitor_1,price_competitor_1,train_type_competitor_1,duration_competitor_1,time_distance_competitor_2,price_competitor_2,train_type_competitor_2,duration_competitor_2
3,00004_01-01-2025-06.57,AVE,414,S-103,2025,1,2,6.95,157.0,82.99,...,AVE,178.0,30.0,27.76,AVE,199.0,60.0,47.82,AVE,157.0
4,00005_01-01-2025-07.02,OUIGO,509,S-108,2025,1,2,7.03,172.0,36.73,...,AVE,157.0,25.2,27.76,AVE,199.0,55.2,47.82,AVE,157.0
5,00002_01-01-2025-07.22,IRYO,461,S-109,2025,1,2,7.37,157.0,30.65,...,AVE,157.0,4.8,27.76,AVE,199.0,34.8,47.82,AVE,157.0
6,00006_01-01-2025-07.27,AVE,414,S-103,2025,1,2,7.45,199.0,27.76,...,AVE,157.0,30.0,47.82,AVE,157.0,60.0,104.56,AVE,172.0
7,00004_01-01-2025-07.57,AVE,414,S-103,2025,1,2,7.95,157.0,47.82,...,AVE,199.0,30.0,104.56,AVE,172.0,60.0,63.01,AVE,164.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15430,00004_31-12-2025-18.57,AVE,414,S-103,2025,12,2,18.95,157.0,96.87,...,AVE,177.0,37.2,25.68,AVLO,201.0,67.8,21.42,AVE,157.0
15431,00009_31-12-2025-19.29,IRYO,461,S-109,2025,12,2,19.48,172.0,34.28,...,AVE,157.0,5.4,25.68,AVLO,201.0,36.0,21.42,AVE,157.0
15432,00019_31-12-2025-19.34,AVLO,581,S-106,2025,12,2,19.57,201.0,25.68,...,AVE,157.0,30.6,21.42,AVE,157.0,52.8,55.58,AVE,201.0
15433,00004_31-12-2025-20.05,AVE,414,S-103,2025,12,2,20.08,157.0,21.42,...,AVLO,201.0,22.2,55.58,AVE,201.0,62.4,62.89,AVE,172.0
